# Similarity and Distance Concepts

**Module:** 01 — Embeddings

Retrieval quality depends on how you measure 'near'. This lesson makes cosine, dot product, Euclidean, and Manhattan distances concrete—and operational.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain semantic similarity vs surface similarity
- Compute and interpret cosine, dot product, Euclidean, and Manhattan metrics
- Choose metrics consistent with vector normalization
- Set and evaluate distance/similarity thresholds for filtering


## Semantic Similarity

**Definition.** Semantic similarity scores how related two items are in **meaning**, independent of exact wording.

**Why it matters.** It is the ranking signal behind semantic search, clustering, and deduplication.

**How it works.** Embed both items, then apply a metric. Higher similarity (or lower distance) ⇒ more related.

**Intuition.** 'Refund window' ≈ 'money-back period' even with different words.

**Common pitfalls.**
- Confusing string similarity (edit distance) with semantic similarity
- Interpreting raw scores across different models without calibration

**When to use.** Any time ranking by meaning beats ranking by keywords.


In [ ]:
# String similarity vs semantic intent (toy)
from difflib import SequenceMatcher

pairs = [
    ("refund policy", "refund policy"),
    ("refund policy", "money back rules"),
    ("refund policy", "red fox policy"),
]
for a, b in pairs:
    surf = SequenceMatcher(None, a, b).ratio()
    print(f"surface={surf:.2f} | {a!r} vs {b!r}")
print("Surface score likes spelling overlap; embeddings should prefer pair 2 over pair 3.")


## Cosine Similarity

**Definition.** Cosine similarity is the cosine of the angle between two vectors: `cos(a,b) = (a·b) / (||a|| ||b||)`, ranging from -1 to 1 (often 0..1 for embeddings).

**Why it matters.** It ignores vector magnitude and focuses on orientation—useful when length encodes document size more than meaning.

**How it works.** L2-normalize vectors, then cosine equals dot product—ANN indexes often assume this.

**Intuition.** Two arrows pointing the same way are similar even if one is longer.

**Common pitfalls.**
- Using cosine on already-dot-trained asymmetric scores without checking model docs
- Forgetting scores are not calibrated probabilities

**When to use.** Default metric for most normalized text embeddings.

| Metric | Sensitive to magnitude? | Common with |
|---|---|---|
| Cosine | No (angle only) | Normalized text embeddings |
| Dot product | Yes | Normalized vectors (≡ cosine) or some dual encoders |
| Euclidean | Yes | Unnormalized / spatial data |
| Manhattan | Yes | Sparse / robust-to-outliers cases |


In [ ]:
import numpy as np

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 4.0, 6.0])  # same direction, different magnitude
c = np.array([3.0, -1.0, 0.0])
print("cos(a,b) same direction:", round(cosine(a, b), 4))
print("cos(a,c) different direction:", round(cosine(a, c), 4))


In [ ]:
# Cosine == dot after L2 normalization
def l2(v):
    return v / (np.linalg.norm(v) + 1e-9)

an, bn = l2(a), l2(b)
print("dot(normalized):", round(float(np.dot(an, bn)), 4))
print("cosine(raw):     ", round(cosine(a, b), 4))


## Dot Product

**Definition.** Dot product `a·b = Σ a_i b_i` measures alignment scaled by magnitudes.

**Why it matters.** Fast, index-friendly, and identical to cosine when vectors are unit-normalized.

**How it works.** Many vector DBs store normalized vectors and use IP (inner product) search.

**Intuition.** If both vectors are long and aligned, dot product is large.

**Common pitfalls.**
- Comparing unnormalized document vectors where longer docs dominate
- Mixing IP and L2 indexes accidentally

**When to use.** When vectors are normalized or your model was trained for inner product.


In [ ]:
# Magnitude inflates dot product
short = np.array([1.0, 0.0])
long = np.array([10.0, 0.0])
query = np.array([1.0, 0.1])
print("dot short", round(float(np.dot(query, short)), 3))
print("dot long ", round(float(np.dot(query, long)), 3), "← wins due to length, not meaning")
print("cosine short", round(cosine(query, short), 3))
print("cosine long ", round(cosine(query, long), 3))


## Euclidean Distance

**Definition.** L2 distance `||a-b||_2 = sqrt(Σ (a_i-b_i)^2)`.

**Why it matters.** Natural when absolute position/magnitude in space is meaningful.

**How it works.** Smaller distance ⇒ closer. Related to cosine on normalized vectors.

**Intuition.** Straight-line distance between two points.

**Common pitfalls.**
- Interpreting L2 on unnormalized text embeddings without care

**When to use.** Image embeddings / spatial features; or when index API defaults to L2.


In [ ]:
a = np.array([0.0, 0.0])
b = np.array([3.0, 4.0])
print("euclidean:", round(float(np.linalg.norm(a - b)), 4))  # 5.0


## Manhattan Distance

**Definition.** L1 distance `Σ |a_i-b_i|` (taxicab distance).

**Why it matters.** More robust to outlier dimensions than L2 in some sparse settings.

**How it works.** Sum absolute coordinate gaps; no squaring.

**Intuition.** City blocks: you cannot cut diagonally through buildings.

**Common pitfalls.**
- Less common as default for dense text embeddings

**When to use.** Certain sparse/learned-sparse or robust-stat scenarios; otherwise prefer cosine.


In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 0.0, 5.0])
print("manhattan:", round(float(np.abs(a - b).sum()), 4))
print("euclidean:", round(float(np.linalg.norm(a - b)), 4))


## Similarity Score

**Definition.** A similarity score is the numeric output you rank on—cosine, reranker logit, or fused hybrid score.

**Why it matters.** Downstream systems need a single ranking key and sometimes a cutoff.

**How it works.** Log scores during eval; calibrate thresholds on a labeled set; do not copy thresholds across models.

**Intuition.** A score is a ranking currency—exchange rates change per model.

**Common pitfalls.**
- Hard-coding 0.8 cosine as 'relevant' for every corpus
- Averaging incompatible metrics

**When to use.** Always define how scores are produced, logged, and thresholded.


In [ ]:
# Rank fusion sketch: min-max normalize then weighted sum
import numpy as np

dense_scores = np.array([0.81, 0.77, 0.62])
bm25_scores = np.array([12.0, 3.5, 9.0])

def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

fused = 0.6 * minmax(dense_scores) + 0.4 * minmax(bm25_scores)
order = np.argsort(-fused)
print("fused scores", np.round(fused, 3))
print("rank order (0-based ids)", order.tolist())


## Distance Threshold

**Definition.** A threshold filters candidates below a similarity floor (or above a distance ceiling) before they reach the LLM or UI.

**Why it matters.** Prevents packing irrelevant context into prompts—a major hallucination source.

**How it works.** Plot score distributions for relevant vs irrelevant pairs; pick a threshold from precision/recall needs; revisit after model changes.

**Intuition.** A bouncer at the club door: only vectors with a high enough score enter.

**Common pitfalls.**
- Static thresholds that silently break after re-embedding
- Thresholding before reranking when score scales differ

**When to use.** RAG grounding, auto-FAQ deflection, duplicate detection.


In [ ]:
# Threshold selection on a tiny labeled set
pairs = [
    ("relevant", 0.86),
    ("relevant", 0.81),
    ("relevant", 0.77),
    ("irrelevant", 0.55),
    ("irrelevant", 0.48),
    ("irrelevant", 0.72),
]

def metrics(threshold):
    tp = sum(1 for y, s in pairs if y == "relevant" and s >= threshold)
    fp = sum(1 for y, s in pairs if y == "irrelevant" and s >= threshold)
    fn = sum(1 for y, s in pairs if y == "relevant" and s < threshold)
    prec = tp / (tp + fp + 1e-9)
    rec = tp / (tp + fn + 1e-9)
    return prec, rec

for t in [0.70, 0.75, 0.80]:
    p, r = metrics(t)
    print(f"threshold={t:.2f}  precision={p:.2f}  recall={r:.2f}")


### Try it yourself — Metric lab

1. Generate 5 random unit vectors and compute cosine and Euclidean matrices.
2. Show that arg-sorting by cosine and by negative Euclidean agree on unit vectors.
3. Propose a thresholding policy for a support bot that prefers precision over recall.


## Deep Dive — Operational Checklist

**Definition.** Production embedding systems are contracts: model ID, dimension, metric, prefixes, and preprocessing must match between index and query paths.

**Why it matters.** Silent contract drift is the most common cause of sudden recall collapse.

**How it works.** Store model metadata with the collection; fail closed on mismatch; re-embed on upgrades with a dual-read window if needed.

**Intuition.** Two maps with different projections cannot share GPS coordinates.

**Common pitfalls.**
- Swapping models without reindexing
- Comparing cosine thresholds across models
- Mixing instruction prefixes inconsistently

**When to use.** Every deployment—not just the first prototype.


In [ ]:
# Contract validator
contract = {
    "model": "text-embedding-3-small",
    "dim": 1536,
    "metric": "cosine",
    "normalize": True,
    "query_prefix": "",
    "doc_prefix": "",
}

def validate_vector(vec, contract):
    assert len(vec) == contract["dim"], (len(vec), contract["dim"])
    return True

validate_vector([0.0] * contract["dim"], contract)
print("contract ok", contract["model"])


### Try it yourself — Contract lab

1. Write the contract dict for your preferred open embedding model.
2. Intentionally break the dimension and show the assertion firing.
3. Document who owns re-embedding after a model upgrade.


In [ ]:
# Threshold calibration sketch
pairs = [("rel", 0.84), ("rel", 0.79), ("irr", 0.55), ("irr", 0.71)]
for thr in [0.70, 0.75, 0.80]:
    tp = sum(1 for y,s in pairs if y=="rel" and s>=thr)
    fp = sum(1 for y,s in pairs if y=="irr" and s>=thr)
    print(thr, "tp", tp, "fp", fp)


## Comparison table — practical choices

| Concern | Prefer | Avoid |
|---|---|---|
| Paraphrase FAQ | Dense / hybrid | Keywords alone |
| Invoice IDs | Lexical / hybrid | Dense-only |
| Privacy VPC | Local open model | Unapproved SaaS |
| Fast prototype | Small API/local MiniLM | Giant untested models |


In [ ]:
# Mini bake-off harness
rankings = {
    "modelA": ["d2", "d1", "d3"],
    "modelB": ["d1", "d2", "d3"],
}
gold = {"d1"}
for name, ranked in rankings.items():
    hit = ranked[0] in gold
    print(name, "top1_hit", hit)


```mermaid
flowchart LR
  A[Corpus] --> B[Embed+index]
  C[Query] --> D[Embed]
  D --> E[Search]
  B --> E
  E --> F[Evaluate recall]
  F -->|bad| G[Fix chunking/model/metric]
  F -->|good| H[Ship with monitors]
```


### Try it yourself — End-to-end

1. Build a 10-document toy index with hashed embeddings.
2. Create 5 paraphrase queries and compute Recall@3.
3. Write one monitoring alert you would page on in production.


In [ ]:
import numpy as np

def emb(text, dim=32):
    rng = np.random.default_rng(abs(hash(text.lower())) % (2**32))
    v = rng.normal(size=dim)
    return v / (np.linalg.norm(v) + 1e-9)

docs = {
    "d1": "Refunds are available for 14 days",
    "d2": "Reset your password via email",
    "d3": "Express shipping takes two days",
}
X = {i: emb(t) for i,t in docs.items()}
q = emb("how long can I return an item?")
ranked = sorted(((float(np.dot(q,v)), i) for i,v in X.items()), reverse=True)
print(ranked)
print("recall@1", float(ranked[0][1] == "d1"))


## Summary & Key Takeaways

- Cosine focuses on angle; prefer it for normalized text embeddings.
- Dot product ≡ cosine on unit vectors; magnitude can distort IP otherwise.
- Thresholds must be calibrated per model and corpus.
- Hybrid fusion needs normalization before combining score lists.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
